# Notebook 1: Merge 3 Datasets

## Set-up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## BarKA-MS

In [ ]:
# Load the dataset
barka = pd.read_csv('../datasets/BarKA_MS_Fitbit_data_anonymized.csv')

# Verify the structure of the BarKA-MS dataset
display(barka.head())
display(barka.info())
display(barka['fakeId'].nunique())

# Process Barka for merging
barka = barka.rename(columns = {
    'fakeId': 'id',
    "fakeTime": "datetime",
    "Steps": "steps"
    })
barka['datetime'] = pd.to_datetime(barka['datetime'], format = 'mixed', errors = 'coerce')
print("Bad datetime rows:", barka['datetime'].isna().sum()) # Verify no errors in datetime conversion

# Create barka_hourly for merging
barka_hourly = (
    barka
    .set_index('datetime')
    .groupby('id')['steps']
    .resample('h')
    .sum()
    .reset_index()
)
barka_hourly['label'] = 1
barka_hourly['source'] = 'barka'

# Verify barka_hourly
display(barka_hourly.head())
display(barka_hourly.info())

## FitBit Fitness Tracker

In [ ]:
# Load the FitBit Kaggle datasets
fitbit_1 = pd.read_csv('../datasets/hourlySteps_merged_31216_41116.csv')
fitbit_2 = pd.read_csv('../datasets/hourlySteps_merged_41216_51216.csv')

# Merge the two FitBit datasets based on common 'Id' values
f1_id = fitbit_1['Id'].unique()
f2_id = fitbit_2['Id'].unique()
same_ids = set(f1_id).intersection(set(f2_id))
fitbit = pd.concat([fitbit_1, fitbit_2], ignore_index = True)
fitbit = fitbit[fitbit['Id'].isin(same_ids)]

# Verify the structure of the FitBit dataset
display(fitbit.head())
display(fitbit.info())
display(fitbit['Id'].nunique())

# Process FitBit for merging
fitbit = fitbit.rename(columns = {
    'Id': 'id',
    'ActivityHour': 'datetime',
    'StepTotal': 'steps'
})
fitbit['datetime'] = pd.to_datetime(
    fitbit['datetime'], 
    format = '%m/%d/%Y %I:%M:%S %p', 
    errors = 'coerce'
    )
fitbit['label'] = 0
fitbit['source'] = 'fitbit'
fitbit['id'] = fitbit['id'].astype(str) # Ensure id is string for merging

# Verify the processed FitBit dataset
display(fitbit.head())
display(fitbit.info())

## LifeSnaps

In [ ]:
# Load the LifeSnaps dataset
lifesnaps = pd.read_csv('../datasets/hourly_fitbit_sema_df_unprocessed.csv')

# Verify the structure of the LifeSnaps dataset
display(lifesnaps.head())
display(lifesnaps.info())
display(lifesnaps['id'].nunique())

# Process LifeSnaps for merging
lifesnaps['datetime'] = pd.to_datetime(lifesnaps['date']) + pd.to_timedelta(lifesnaps['hour'], unit = 'h')
lifesnaps_hourly = lifesnaps[['id', 'datetime', 'steps']].copy()
lifesnaps_hourly['steps'] = lifesnaps_hourly['steps'].fillna(0) # Fill missing steps with 0
lifesnaps_hourly['label'] = 0
lifesnaps_hourly['source'] = 'lifesnaps'

# Verify the processed LifeSnaps dataset
display(lifesnaps_hourly.head())
display(lifesnaps_hourly.info())


# Merge Datasets

In [ ]:

MAX_HOURS = 720 # 30 days * 24 hours | Cap contribution from each dataset
combined_data = pd.concat([barka_hourly, fitbit, lifesnaps_hourly], ignore_index = True)
combined_data = combined_data.sort_values(['id', 'datetime']).reset_index(drop = True)
combined_data = (
    combined_data
    .groupby('id')
    .head(MAX_HOURS) 
    .reset_index(drop = True)
)
display(combined_data.head())
display(combined_data.info())

# Save the combined dataset
combined_data.to_csv('../datasets/combined_activity_data.csv', index = False)


In [ ]:
summary = (
    combined_data.groupby('source')
    .agg(
        Class = ('label', lambda x: 'MS' if x.iloc[0] == 1 else 'Healthy'),
        Participants = ('id', 'nunique'),
        Hours = ('id', 'size')
    )
    .reset_index()
    .rename(columns={'source': 'Source'})
)

summary['Participant %'] = (summary['Participants'] / summary['Participants'].sum() * 100).round(1)

summary.loc[len(summary)] = [
    'Total',
    '-',
    summary['Participants'].sum(),
    summary['Hours'].sum(),
    100.0
]

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

tbl = ax.table(
    cellText = summary.values,
    colLabels = summary.columns,
    cellLoc = 'center',
    loc = 'center'
)

tbl.scale(2, 2)
for col in range(len(summary.columns)):
    tbl[(0, col)].get_text().set_weight('bold')
for row in range(1, len(summary) + 1):
    tbl[(row, 0)].get_text().set_weight('bold')

plt.tight_layout()
plt.savefig('../output/dataset_summary_table.png', dpi=300, bbox_inches='tight')
plt.show()